---
title: "Part 6: Grammar of Graphics with Lets-Plot"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/06-lets-plot.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/06-lets-plot.ipynb)

**DS-MLOps Python Foundations**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook assumes you have completed Part 5 (`05-matplotlib.ipynb`). If you have not, start there: this notebook rebuilds several of its charts on purpose, so the contrast between the two ways of thinking about a plot is concrete rather than abstract.

Part 5 was **imperative**: you told matplotlib exactly which method to call, in which order, for every piece of the chart. This part is **declarative**: you describe the data and the mapping you want, and the library works out how to draw it. This style is called the **grammar of graphics**, and **lets-plot** implements it in Python the way ggplot2 implements it in R. Part 7 (`07-data-storytelling.ipynb`) covers what makes a chart good and applies this project's house style to both libraries.

::: {.callout-note collapse="true" icon=false}
## Topics covered

| Topic | Why it matters |
|---|---|
| **Declarative vs. imperative** | The mental model shift that makes the rest of this notebook click |
| **`ggplot` + `aes` + `geom`** | The three pieces every lets-plot chart is built from |
| **Mapping vs. setting** | The single most common mistake in any grammar of graphics |
| **Layering** | Adding a statistical summary on top of raw data, declaratively |
| **Faceting** | One line to replace a whole manual subplot loop |
| **Titles, labels, and scales** | Communicating clearly: naming axes, renaming legends, controlling colours |
:::

> Callout markers used throughout this notebook are explained on the [book cover page](../../index.qmd#callout-guide).

## Why Grammar of Graphics?

You have already drawn scatter plots with Matplotlib. You called `ax.scatter(x, y, c=color, s=size)`. If you wanted a different geom, you called a different method. If you wanted a trend line, you called yet another. The plot grew by accumulating function calls, each one doing something slightly different.

There is another way to think about it. A chart is not a list of drawing commands: it is a **mapping** from data columns to visual channels (position, colour, shape, size). State that mapping once, and the library figures out how to draw it. Add a layer (a trend line, a rug, a label) and you extend the mapping rather than calling a new drawing function. This is the Grammar of Graphics, introduced by Leland Wilkinson in 1999 and popularised by Hadley Wickham's **ggplot2** for R.

**Lets-Plot** ([lets-plot.org](https://lets-plot.org)) is JetBrains' Python implementation of the same grammar. Its API mirrors ggplot2 so closely that R users can read Lets-Plot code without a translation guide. It renders to HTML in Jupyter, to PNG for reports, and (in its Pro edition) to interactive Datalore dashboards.

### Alternatives that use the same grammar

| Library | Language | Notes |
| --- | --- | --- |
| **ggplot2** ([ggplot2.tidyverse.org](https://ggplot2.tidyverse.org)) | R | The original; Lets-Plot mirrors it deliberately |
| **plotnine** ([plotnine.org](https://plotnine.org)) | Python | ggplot2 port; similar API, Matplotlib backend |
| **Lets-Plot** ([lets-plot.org](https://lets-plot.org)) | Python / Kotlin | HTML-first output, fast, maintained by JetBrains |
| **Vega-Altair** ([altair-viz.github.io](https://altair-viz.github.io)) | Python | Different grammar (Vega-Lite), interactive |

### Already in your environment

```bash
uv add lets-plot          # for a standalone project
```

::: {.callout-note collapse="true" icon=false}
## Learning Objectives

By the end of Part 6 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain the difference between declarative and imperative plotting | Sec. 1 |
| 2 | Build a chart from `ggplot()`, `aes()`, and a `geom_*()` | Sec. 2 |
| 3 | Distinguish mapping a variable from setting a fixed value | Sec. 3 |
| 4 | Layer a statistical summary on top of raw data | Sec. 4 |
| 5 | Facet one chart into many panels instead of looping over subplots | Sec. 5 |
| 6 | Add titles, axis labels, and control colours with `labs()` and scale functions | Sec. 6 |
:::


## 1. Declarative vs. Imperative

In Part 5, building a scatter plot meant calling `ax.scatter()` directly: you were the one deciding which function draws which shape. The grammar of graphics flips this around. You describe **what the data means** (this column is the x position, this column is the colour) and the library decides how to draw it. The same description works whether you add one point or one million, and stays valid even if you later change the chart type entirely.

In [ ]:
from lets_plot import LetsPlot

LetsPlot.setup_html()

Rebuild the same dataset from Part 5: exam results across three courses and two semesters.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

courses = np.array(["Machine Learning", "Data Structures", "Statistics"])
semesters = np.array(["Fall 2024", "Spring 2025"])

n_per_group = 60
course_col = np.repeat(courses, n_per_group * len(semesters))
semester_col = np.tile(np.repeat(semesters, n_per_group), len(courses))

course_base = {"Machine Learning": 68, "Data Structures": 74, "Statistics": 71}
semester_bump = {"Fall 2024": 0, "Spring 2025": 4}

exam_score = np.array(
    [rng.normal(course_base[c] + semester_bump[s], 10) for c, s in zip(course_col, semester_col, strict=True)]
).clip(0, 100)
study_hours = rng.uniform(0, 25, size=len(course_col))

results = pd.DataFrame(
    {
        "course": course_col,
        "semester": semester_col,
        "exam_score": exam_score,
        "study_hours": study_hours,
    }
)
results.head()

Here is the Part 5 scatter plot (study hours vs. exam score) again, this time declaratively:

In [ ]:
from lets_plot import aes, geom_point, ggplot

ggplot(results, aes(x="study_hours", y="exam_score")) + geom_point(alpha=0.4)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: The Grammar of Graphics</span><br><br>
Every lets-plot chart is built from the same three pieces, combined with <code>+</code>: <code>ggplot(data, aes(...))</code> declares the dataset and which columns map to which visual property, and one or more <code>geom_*()</code> layers say what shape to draw with that mapping. Change the <code>geom_*()</code> and the same <code>aes()</code> mapping produces a completely different chart type, often without touching anything else.
</div>

## 2. The Grammar: `ggplot`, `aes`, `geom`

`aes()` (short for "aesthetics") maps DataFrame columns to visual properties: `x`, `y`, `color`, `fill`, `size`. The `geom_*()` you add decides what shape represents each row. Swapping `geom_point()` for `geom_line()` or `geom_bar()` is often the only change needed to turn one chart type into another:

In [ ]:
from lets_plot import geom_bar

course_means = results.groupby("course", as_index=False)["exam_score"].mean()

ggplot(course_means, aes(x="course", y="exam_score")) + geom_bar(stat="identity", fill="#4477AA")

`stat="identity"` tells `geom_bar()` to plot the `exam_score` column exactly as given, rather than its default behaviour of counting rows per category. Compare this to Part 5's bar chart: the data preparation (`groupby().mean()`) is identical, only the drawing step changed shape.

`position="dodge"` places bars for each group side by side instead of stacking them, useful when you want to compare values across two grouping variables at once. The data needs a category column on `x` and a group column on `fill`:

In [ ]:
course_semester_means = results.groupby(["course", "semester"], as_index=False)["exam_score"].mean()

(
    ggplot(course_semester_means, aes(x="course", y="exam_score", fill="semester"))
    + geom_bar(stat="identity", position="dodge")
)

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Histogram, Declaratively</span><br><br>

<b>Goal:</b> Rebuild the Part 5 histogram of <code>exam_score</code> using <code>geom_histogram()</code>. Map <code>x="exam_score"</code> in <code>aes()</code>, and pass <code>bins=20</code> to <code>geom_histogram()</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>ggplot(results, aes(x="exam_score")) + geom_histogram(bins=20, fill="#4477AA")</pre>
</div>

In [ ]:
# TODO: build the histogram described above
...

## 3. Mapping vs. Setting

`aes(color="course")` **maps** the `course` column to colour: each course gets its own colour, chosen automatically, with a legend. `color="#4477AA"` **sets** every point to the exact same fixed colour, with no legend, because there is nothing left to distinguish. Confusing the two is the single most common mistake when learning any grammar of graphics, in Python or R:

In [ ]:
# Mapping: course determines colour, one colour per course, with a legend
ggplot(results, aes(x="study_hours", y="exam_score", color="course")) + geom_point(alpha=0.6)

In [ ]:
# Setting: every point is the same fixed colour, no legend needed
ggplot(results, aes(x="study_hours", y="exam_score")) + geom_point(color="#4477AA", alpha=0.6)

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Putting a fixed value inside <code>aes()</code></span><br><br>
<code>aes(color="#4477AA")</code> tries to map the literal string <code>"#4477AA"</code> as if it were a column name. Lets-plot will either error or, worse, silently treat it as a constant category and draw a one-entry legend that says <code>#4477AA</code>. A fixed value is a <b>setting</b> and belongs as a plain keyword argument to the <code>geom_*()</code>, outside <code>aes()</code>. A column name that should vary per row is a <b>mapping</b> and belongs inside <code>aes()</code>.
</div>

## 4. Layering: Raw Data and a Statistical Summary Together

Because every `geom_*()` is its own layer, you can stack a raw-data layer and a computed-summary layer on the same `aes()` mapping. `geom_smooth()` fits a trend line to the data it is given, with a shaded confidence band, entirely declaratively:

In [ ]:
from lets_plot import geom_smooth

(
    ggplot(results, aes(x="study_hours", y="exam_score"))
    + geom_point(alpha=0.3, color="#4477AA")
    + geom_smooth(color="#EE6677", se=True)
)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Use <code>method='lm'</code> when you need a straight-line fit</span><br><br>
<code>geom_smooth()</code> uses a LOESS (locally weighted) smoother by default, which follows local curves in the data. When you specifically want a linear regression fit, pass <code>method='lm'</code>. The confidence band and <code>se=True/False</code> control both methods the same way:

<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'># Default LOESS: follows curves
geom_smooth(se=True)

# Linear (OLS): forces a straight line, shows the parametric fit
geom_smooth(method='lm', se=True)</pre>

The LOESS smoother needs <code>statsmodels</code> installed; the linear one does not. Both display the 95% confidence band when <code>se=True</code>.
</div>

`geom_density()` is the smooth-curve equivalent of a histogram, useful when comparing several distributions that would otherwise overlap into an unreadable stack of bars:

In [ ]:
from lets_plot import geom_density

ggplot(results, aes(x="exam_score", fill="course")) + geom_density(alpha=0.4)

`geom_boxplot()` summarises a distribution as median, quartiles, and outliers per group, the declarative equivalent of Part 5's `sns.boxplot()`. Because it is just another `geom_*()`, it composes with `aes()` and faceting exactly like any other layer:

In [ ]:
from lets_plot import geom_boxplot

ggplot(results, aes(x="course", y="exam_score", fill="course")) + geom_boxplot(alpha=0.7)

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Swap <code>geom_boxplot()</code> for <code>geom_violin()</code> when shape matters</span><br><br>
A boxplot gives you five numbers: minimum, first quartile, median, third quartile, and maximum. A violin gives you the full density shape on both sides of the axis, which reveals skew or multiple peaks that the box hides. The <code>aes()</code> mapping stays identical: just change the <code>geom_*()</code>.
</div>

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Layers compose in the order you add them</span><br><br>
<code>geom_point() + geom_smooth()</code> draws points first, then the trend line on top. Reversing the order draws the line first and the points on top of it. This matters most when a layer has a solid fill that would otherwise hide whatever was drawn before it.
</div>

<div style='background:#F5F3FF;border-left:5px solid #7C3AED;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#5B21B6;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Add marginal distribution plots with <code>ggmarginal()</code></span><br><br>
Lets-Plot's <code>ggmarginal()</code> wraps any scatter plot with histograms or density curves along the x and y margins, letting you see both the bivariate relationship and the individual distributions without a separate figure:

<pre style='background:#F4F5F6;padding:10px;border-radius:4px;font-size:0.9em'>from lets_plot import ggmarginal

p = (
    ggplot(results, aes(x="study_hours", y="exam_score", color="course"))
    + geom_point(alpha=0.4)
    + geom_smooth(method="lm", se=False)
)
ggmarginal(p, type="density")</pre>

<code>type</code> accepts <code>"density"</code> (KDE curves), <code>"histogram"</code>, or <code>"boxplot"</code>. The marginals automatically inherit the <code>color</code> mapping so each course gets its own density curve.
</div>

## 5. Faceting: One Plot, Many Panels

Part 5's three-panel histogram needed a manual loop over `axes.flat`, plus `sharey=True` to keep the comparison fair. `facet_wrap()` does both in one line, and **shares scales across panels by default**, the opposite of matplotlib's default and exactly what you want for a fair comparison:

In [ ]:
from lets_plot import facet_wrap, geom_histogram, ggsize

(
    ggplot(results, aes(x="exam_score"))
    + geom_histogram(bins=15, fill="#4477AA")
    + facet_wrap(facets="course")
    + ggsize(700, 250)
)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Faceting replaces a loop with a declaration</span><br><br>
<code>facet_wrap(facets="course")</code> splits the data by the <code>course</code> column and draws one panel per group, using the exact same <code>aes()</code> and <code>geom_*()</code> for every panel. There is no loop to write and no risk of accidentally giving one panel a different scale than the others, the bug from Part 5's Common Mistake callout. Add a second facet variable with <code>facet_grid()</code> for a full grid of panels.
</div>

When you have two grouping variables, `facet_grid()` builds a full grid of panels: one row per level of one variable and one column per level of the other. For this dataset, rows are semesters and columns are courses:

In [ ]:
from lets_plot import facet_grid

(
    ggplot(results, aes(x="exam_score"))
    + geom_histogram(bins=12, fill="#4477AA")
    + facet_grid(y="semester", x="course")
    + ggsize(850, 450)
)

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 2 - Facet by Semester</span><br><br>

<b>Goal:</b> Build a scatter plot of <code>study_hours</code> vs. <code>exam_score</code>, coloured by <code>course</code>, faceted into one panel per <code>semester</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>ggplot(results, aes(x="study_hours", y="exam_score", color="course")) \
    + geom_point(alpha=0.5) \
    + facet_wrap(facets="semester")</pre>
</div>

In [ ]:
# TODO: scatter plot, coloured by course, faceted by semester
...

## 6. Titles, Labels, and Scales

Every chart so far has used whatever axis labels and legend titles lets-plot generates by default: column names from the DataFrame, which are fine for exploration but not for sharing. `labs()` replaces all of them in one layer. `scale_color_manual()` and `scale_fill_manual()` give you exact control over which colour maps to which group. Passing `pro_colors` from `ark.plot.theme` uses the project's brand palette, the same one `modern_theme()` applies globally in Part 7:

In [ ]:
from lets_plot import labs, scale_color_manual

from ark.plot.theme import pro_colors

(
    ggplot(results, aes(x="study_hours", y="exam_score", color="course"))
    + geom_point(alpha=0.5)
    + geom_smooth(se=False)
    + labs(
        title="Study hours versus exam score",
        subtitle="Each point is one student; lines show the per-course trend",
        x="Weekly study hours",
        y="Exam score (0-100)",
        color="Course",
    )
    + scale_color_manual(values=pro_colors)
)

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>labs()</code> renames anything auto-generated from a column name</span><br><br>
<code>labs(title=..., x=..., y=..., color=...)</code> is one more <code>+</code> layer. The named argument matches the aesthetic: use <code>color=</code> when you have <code>aes(color="course")</code> and <code>fill=</code> when you have <code>aes(fill="course")</code>. <code>pro_colors</code> from <code>ark.plot.theme</code> is the project's brand palette; passing it to <code>scale_color_manual()</code> connects this chart to the same colour system that <code>modern_theme()</code> applies globally in Part 7.
</div>

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 3 - Label the Density Chart</span><br><br>

<b>Goal:</b> Take the <code>geom_density()</code> chart from Section 4 and add a proper title, axis labels, and legend title with <code>labs()</code>.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>ggplot(results, aes(x="exam_score", fill="course")) + geom_density(alpha=0.4) \
    + labs(title=..., x=..., y=..., fill=...)</pre>
</div>

In [ ]:
# TODO: add labs() to the density chart from Section 4
...

## Capstone: The Three-Panel Report, Declaratively

Part 5's capstone built a three-panel course report with a manual `(1, 3)` grid of Axes. Here, build the same idea (one histogram per course) as a **single faceted chart** instead of three separate panels assembled by hand.

<div style='background:#EBF5F0;border-left:5px solid #009E73;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#065F46;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Faceted Course Report</span><br><br>

<b>Goal:</b> Build one chart: a histogram of <code>exam_score</code>, faceted by <code>course</code>, with <code>geom_vline()</code> marking the overall mean score so each course's panel can be compared against it.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>overall_mean = results["exam_score"].mean()

ggplot(results, aes(x="exam_score")) \
    + geom_histogram(bins=15, fill="#4477AA") \
    + geom_vline(xintercept=overall_mean, color="#EE6677", linetype="dashed") \
    + facet_wrap(facets="course") \
    + ggsize(700, 250)</pre>
<b>Hint:</b> <code>geom_vline()</code> takes a fixed <code>xintercept</code>, a setting, not a mapping, so it goes outside <code>aes()</code> just like the fixed colours in Sec. 3.
</div>

In [ ]:
from lets_plot import geom_vline

overall_mean = results["exam_score"].mean()

# TODO: faceted histogram with a reference line at overall_mean
...

## Further Reading

| Resource | Why it matters |
|---|---|
| Wilkinson, L. (2005). *The Grammar of Graphics*, 2nd ed. Springer. | The theory behind the layered grammar; Lets-Plot, ggplot2, and Vega-Altair are all implementations of this framework |
| Wickham, H. (2010). [A layered grammar of graphics](https://doi.org/10.1198/jcgs.2009.07098). *Journal of Computational and Graphical Statistics* 19(1), 3–28. | The ggplot2 paper; most directly explains `geom_*`, `aes()`, and `stat_*` concepts that Lets-Plot mirrors |
| [Lets-Plot documentation](https://lets-plot.org) | The primary API reference; the gallery is the fastest way to find the right `geom_*` |
| [ggplot2 book](https://ggplot2-book.org) (Wickham, 2016) | Free online: Lets-Plot's API maps closely to ggplot2, so this book is directly useful |


## Summary

| Concept | Key rule |
|---|---|
| Declarative vs. imperative | Describe the mapping, let the library decide how to draw it |
| `ggplot(data, aes(...))` | Declares the dataset and which columns map to which visual property |
| `geom_*()` | Decides what shape represents each row; swap it to change chart type |
| Mapping | `aes(color="course")`, a column that varies per row, gets a legend |
| Setting | `color="#4477AA"`, a fixed value, no legend |
| `position="dodge"` | Places bars for each group side by side instead of stacking them |
| Layering | Stack `geom_*()` calls with `+`; later layers draw on top of earlier ones |
| `geom_smooth()` / `geom_density()` | Statistical summaries, computed declaratively from raw data |
| `geom_boxplot()` / `geom_violin()` | Distribution per group: box for five-number summary, violin for full shape |
| `facet_wrap()` | One declaration replaces a manual subplot loop, with shared scales by default |
| `facet_grid(rows=..., cols=...)` | Full grid of panels for two grouping variables |
| `labs()` | Replaces any auto-generated axis label or legend title in one layer |
| `scale_color_manual()` | Maps groups to specific colours, overriding the default palette |

**Next:** `07-data-storytelling.ipynb`, covering what makes a chart good and applying this project's house style to both matplotlib and lets-plot.